<a href="https://colab.research.google.com/github/ABDULLAH-104/Redit-Post-Classifier/blob/main/IDS_ASSIGNMENT_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PART : 1**

```

```



In [ ]:
# Install libraries
!pip install pandas faker -q

import pandas as pd
import random
import datetime
from faker import Faker

fake = Faker()
random.seed(42)

# ── Realistic Reddit post templates ──────────────────────
tech_titles = [
    "GPT-5 reportedly achieves 99% accuracy on benchmark tests",
    "Apple announces new M4 chip with 40% performance boost",
    "Major data breach exposes 200 million user records",
    "Open-source AI model beats proprietary systems in new study",
    "EU passes landmark AI regulation bill into law",
    "Google fires 200 engineers after AI automation rollout",
    "Tesla FSD version 13 achieves Level 4 autonomy in testing",
    "New quantum computer breaks encryption record",
    "Meta releases open-source LLM rivaling GPT-4",
    "Cybersecurity firm warns of critical zero-day vulnerability",
    "SpaceX Starlink reaches 5 million subscribers worldwide",
    "Microsoft integrates Copilot into every Windows app",
    "OpenAI launches $20B fundraising round at $300B valuation",
    "Scientists use AI to discover 3 new cancer treatments",
    "Reddit API changes force third-party apps to shut down",
    "Deepfake detection accuracy drops to 50% in new study",
    "Amazon lays off 18,000 in AWS restructuring",
    "New study: screen time has no measurable effect on IQ",
    "Linux kernel reaches 30 million lines of code milestone",
    "Self-driving trucks now operating on 10 US highways",
    "Bitcoin ETF approved — institutional demand surges",
    "Solar panel efficiency hits new record at 47.6%",
    "GitHub Copilot now writes 40% of code at Microsoft",
    "NASA telescope captures deepest image of universe ever",
    "TikTok ban signed into law — what happens next?",
]

mental_titles = [
    "I finally told my therapist everything and it changed my life",
    "Does anyone else feel completely numb and don't know why?",
    "Tips that actually helped my anxiety (not just breathe deeply)",
    "3 years clean from self-harm — sharing my story",
    "My depression makes me feel like a burden to everyone",
    "How do I support a partner with bipolar disorder?",
    "Medication made me feel like a zombie — is this normal?",
    "I have been in therapy for 2 years and finally feel progress",
    "Panic attack at work — how did you handle yours?",
    "ADHD diagnosis at 32 explained so much about my life",
    "The intrusive thoughts are getting louder lately",
    "Has anyone successfully come off antidepressants?",
    "Social anxiety has cost me every relationship I have had",
    "I think I am finally ready to ask for help",
    "Grief after losing my dad — how long does it last?",
    "DBT therapy literally saved my life — AMA",
    "Why is mental health care so inaccessible for poor people?",
    "Burnout vs depression — how do you tell the difference?",
    "My therapist said I have complex PTSD — what now?",
    "Small wins thread — share something positive today",
    "Journaling changed my mental health more than therapy did",
    "I cancelled therapy because I cannot afford $200 per session",
    "Mindfulness felt fake until I actually committed to it",
    "Telling my boss about my anxiety was the best decision",
    "Survivors guilt after losing a friend to suicide",
]

def generate_posts(subreddit, titles, n=25):
    posts = []
    flairs = {
        "technology": ["AI","Security","Policy","Hardware","Software","Science"],
        "mentalhealth": ["Support","Advice","Recovery","Discussion","Resources","Vent"]
    }
    for i in range(n):
        upvotes = random.randint(100, 45000) if subreddit=="technology" else random.randint(50, 8000)
        ts = datetime.datetime(2024, 1, 1) + datetime.timedelta(days=random.randint(0, 364),
                                                                  hours=random.randint(0, 23))
        posts.append({
            "subreddit"    : subreddit,
            "post_id"      : fake.lexify("??????", letters="abcdefghijklmnopqrstuvwxyz0123456789"),
            "title"        : titles[i % len(titles)],
            "body"         : fake.paragraph(nb_sentences=3),
            "author"       : fake.user_name(),
            "num_comments" : int(upvotes * random.uniform(0.02, 0.35)),
            "upvotes"      : upvotes,
            "upvote_ratio" : round(random.uniform(0.72, 0.99), 2),
            "url"          : f"https://reddit.com/r/{subreddit}/comments/{fake.lexify('??????')}/",
            "timestamp"    : ts.strftime("%Y-%m-%d %H:%M:%S"),
            "flair"        : random.choice(flairs[subreddit]),
        })
    return posts

# Generate data
all_posts = generate_posts("technology", tech_titles, 25)
all_posts += generate_posts("mentalhealth", mental_titles, 25)

# Build DataFrame
df = pd.DataFrame(all_posts)

# Save CSV
df.to_csv("raw_reddit_data.csv", index=False, encoding="utf-8-sig")

# Download it
from google.colab import files
files.download("raw_reddit_data.csv")

print(f"✓ Done! {len(df)} posts saved")
print(df[["subreddit","title","upvotes","num_comments"]].head(10).to_string(index=False))

In [ ]:
!pip install pandas textblob scikit-learn -q
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
print("✓ All libraries ready")

In [ ]:
import pandas as pd

df = pd.read_csv("raw_reddit_data.csv")
print(f"✓ Loaded {len(df)} posts")
df.head()

In [ ]:
from textblob import TextBlob

def get_sentiment(text):
    score = TextBlob(str(text)).sentiment.polarity
    if score > 0.1:
        return "Positive"
    elif score < -0.1:
        return "Negative"
    else:
        return "Neutral"

def get_sentiment_score(text):
    return round(TextBlob(str(text)).sentiment.polarity, 3)

df["sentiment"]       = df["title"].apply(get_sentiment)
df["sentiment_score"] = df["title"].apply(get_sentiment_score)

print("✓ Sentiment done")
print(df["sentiment"].value_counts())

In [ ]:
def classify_topic(title, subreddit):
    title = title.lower()

    if subreddit == "technology":
        if any(w in title for w in ["ai", "gpt", "llm", "model", "openai", "copilot", "deepfake"]):
            return "Artificial Intelligence"
        elif any(w in title for w in ["hack", "breach", "security", "vulnerability", "zero-day"]):
            return "Cybersecurity"
        elif any(w in title for w in ["law", "ban", "regulation", "policy", "eu", "government"]):
            return "Tech Policy"
        elif any(w in title for w in ["chip", "quantum", "hardware", "solar", "battery"]):
            return "Hardware & Science"
        elif any(w in title for w in ["layoff", "fired", "job", "restructur"]):
            return "Business & Jobs"
        else:
            return "General Technology"

    elif subreddit == "mentalhealth":
        if any(w in title for w in ["anxiety", "panic", "stress", "worry"]):
            return "Anxiety"
        elif any(w in title for w in ["depress", "numb", "sad", "burden", "hopeless"]):
            return "Depression"
        elif any(w in title for w in ["therapy", "therapist", "dbt", "cbt", "medication"]):
            return "Therapy & Treatment"
        elif any(w in title for w in ["adhd", "ptsd", "bipolar", "diagnosis"]):
            return "Mental Health Conditions"
        elif any(w in title for w in ["recover", "clean", "progress", "better", "win"]):
            return "Recovery & Support"
        else:
            return "General Mental Health"

df["topic"] = df.apply(lambda row: classify_topic(row["title"], row["subreddit"]), axis=1)

print("✓ Topic classification done")
print(df["topic"].value_counts())

In [ ]:
def engagement_level(upvotes):
    if upvotes >= 10000:
        return "Viral"
    elif upvotes >= 1000:
        return "High"
    elif upvotes >= 100:
        return "Medium"
    else:
        return "Low"

df["engagement_level"] = df["upvotes"].apply(engagement_level)

print("✓ Engagement levels added")
print(df["engagement_level"].value_counts())

In [ ]:
print(f"Final dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df[["subreddit","title","sentiment","topic","engagement_level","upvotes"]].head(10)

In [ ]:
df.to_csv("classified_reddit_data.csv", index=False, encoding="utf-8-sig")

from google.colab import files
files.download("classified_reddit_data.csv")

print(f"✓ Saved classified_reddit_data.csv with {len(df)} rows and {len(df.columns)} columns")

**PART 2**

In [ ]:
!pip install nltk spacy -q
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

print("✓ All libraries ready")

In [ ]:
import pandas as pd

df = pd.read_csv("raw_reddit_data.csv")
print(f"✓ Loaded {len(df)} posts")
print(df["title"].head(3))

In [ ]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    # Step 1 - Lowercase
    text = text.lower()

    # Step 2 - Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Step 3 - Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Step 4 - Remove numbers
    text = re.sub(r"\d+", "", text)

    # Step 5 - Tokenize
    tokens = word_tokenize(text)

    # Step 6 - Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Step 7 - Remove short words (less than 2 characters)
    tokens = [word for word in tokens if len(word) > 2]

    # Step 8 - Join back to string
    cleaned = " ".join(tokens)

    return cleaned

print("✓ Preprocessing function defined")

# Quick test
sample = "GPT-5 reportedly achieves 99% accuracy on benchmark tests!!"
print(f"\nOriginal : {sample}")
print(f"Cleaned  : {preprocess_text(sample)}")

In [ ]:
# Apply to title and body columns
df["cleaned_title"] = df["title"].apply(preprocess_text)
df["cleaned_body"]  = df["body"].apply(preprocess_text)

# Add word count before and after
df["words_before"] = df["title"].apply(lambda x: len(x.split()))
df["words_after"]  = df["cleaned_title"].apply(lambda x: len(x.split()))

print("✓ Preprocessing applied to all posts")
print(f"   Columns now: {list(df.columns)}")

In [ ]:
from IPython.display import display, HTML

# Pick 10 samples (5 from each subreddit)
tech_samples   = df[df["subreddit"] == "technology"].head(5)
mental_samples = df[df["subreddit"] == "mentalhealth"].head(5)
samples        = pd.concat([tech_samples, mental_samples]).reset_index(drop=True)

# Build comparison table
comparison = pd.DataFrame({
    "No."         : range(1, 11),
    "Subreddit"   : samples["subreddit"],
    "Original Title"  : samples["title"],
    "Cleaned Title"   : samples["cleaned_title"],
    "Words Before": samples["words_before"],
    "Words After" : samples["words_after"],
})

print("=" * 70)
print("   FIGURE 2: Preprocessed Text Samples")
print("=" * 70)

pd.set_option("display.max_colwidth", 50)
display(comparison)

print(f"\nAverage words before cleaning : {df['words_before'].mean():.1f}")
print(f"Average words after cleaning  : {df['words_after'].mean():.1f}")
print(f"Average words removed         : {(df['words_before'] - df['words_after']).mean():.1f}")

In [ ]:
df.to_csv("raw_UPDATED_reddit_data.csv", index=False, encoding="utf-8-sig")

from google.colab import files
files.download("raw_UPDATED_reddit_data.csv")

print(f"✓ Updated CSV saved with {df.shape[0]} rows × {df.shape[1]} columns")
print(f"✓ New columns added: cleaned_title, cleaned_body, words_before, words_after")

**PART 3:**

In [ ]:
!pip install textblob vaderSentiment scikit-learn -q

import nltk
nltk.download('vader_lexicon')

print("✓ All libraries ready")

In [ ]:
import pandas as pd

df = pd.read_csv("raw_reddit_data.csv")
print(f"✓ Loaded {len(df)} posts")
print(f"Columns: {list(df.columns)}")

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = analyzer.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

def get_sentiment_score(text):
    return round(analyzer.polarity_scores(str(text))["compound"], 4)

# Apply to cleaned title
df["sentiment"]       = df["cleaned_title"].apply(get_sentiment)
df["sentiment_score"] = df["cleaned_title"].apply(get_sentiment_score)

print("✓ Sentiment classification done")
print(df["sentiment"].value_counts())

In [ ]:
def classify_topic(title, subreddit):
    title = str(title).lower()

    if subreddit == "technology":
        if any(w in title for w in ["ai","gpt","llm","model","openai",
                                     "copilot","deepfake","machine learning"]):
            return "Artificial Intelligence"
        elif any(w in title for w in ["hack","breach","security",
                                       "vulnerability","zero-day","cyber"]):
            return "Cybersecurity"
        elif any(w in title for w in ["law","ban","regulation","policy",
                                       "eu","government","bill"]):
            return "Tech Policy"
        elif any(w in title for w in ["chip","quantum","hardware",
                                       "solar","battery","telescope","nasa"]):
            return "Hardware & Science"
        elif any(w in title for w in ["layoff","fired","job",
                                       "restructur","employ"]):
            return "Business & Jobs"
        else:
            return "General Technology"

    elif subreddit == "mentalhealth":
        if any(w in title for w in ["anxiety","panic","stress","worry","fear"]):
            return "Anxiety"
        elif any(w in title for w in ["depress","numb","sad",
                                       "burden","hopeless","guilt"]):
            return "Depression"
        elif any(w in title for w in ["therapy","therapist","dbt",
                                       "cbt","medication","antidepressant"]):
            return "Therapy & Treatment"
        elif any(w in title for w in ["adhd","ptsd","bipolar",
                                       "diagnosis","condition"]):
            return "Mental Health Conditions"
        elif any(w in title for w in ["recover","clean","progress",
                                       "better","win","positive","saved"]):
            return "Recovery & Support"
        else:
            return "General Mental Health"

df["topic"] = df.apply(
    lambda row: classify_topic(row["cleaned_title"], row["subreddit"]), axis=1
)

print("✓ Topic classification done")
print(df["topic"].value_counts())

In [ ]:
def engagement_level(upvotes):
    if upvotes >= 10000:
        return "Viral"
    elif upvotes >= 1000:
        return "High"
    elif upvotes >= 100:
        return "Medium"
    else:
        return "Low"

df["engagement_level"] = df["upvotes"].apply(engagement_level)

print("✓ Engagement level added")
print(df["engagement_level"].value_counts())

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# ── Create ground truth labels manually ───────────────────
# Based on subreddit: technology posts lean Neutral/Negative
# mentalhealth posts lean Negative/Neutral
# We define simple rule-based ground truth for evaluation

def ground_truth(title, subreddit):
    title = str(title).lower()
    if subreddit == "technology":
        if any(w in title for w in ["breach","layoff","ban","fired",
                                     "deepfake","vulnerability","drops"]):
            return "Negative"
        elif any(w in title for w in ["achieves","record","approved",
                                       "new","boost","discover"]):
            return "Positive"
        else:
            return "Neutral"
    else:
        if any(w in title for w in ["depress","numb","burden","guilt",
                                     "anxiety","cancel","cost","louder"]):
            return "Negative"
        elif any(w in title for w in ["progress","saved","recover",
                                       "clean","better","positive","ready"]):
            return "Positive"
        else:
            return "Neutral"

df["ground_truth"] = df.apply(
    lambda row: ground_truth(row["title"], row["subreddit"]), axis=1
)

# ── Accuracy Score ─────────────────────────────────────────
acc = accuracy_score(df["ground_truth"], df["sentiment"])
print(f"\n✓ Accuracy Score : {acc:.2%}")

# ── Classification Report ──────────────────────────────────
print("\nClassification Report:")
print(classification_report(df["ground_truth"], df["sentiment"],
                             target_names=["Negative","Neutral","Positive"]))

# ── Confusion Matrix ───────────────────────────────────────
labels = ["Positive", "Neutral", "Negative"]
cm     = confusion_matrix(df["ground_truth"], df["sentiment"], labels=labels)

plt.figure(figsize=(7, 5))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=labels,
            yticklabels=labels,
            linewidths=0.5)

plt.title("Figure 3: Confusion Matrix — Sentiment Classification", fontsize=13)
plt.xlabel("Predicted Label",  fontsize=11)
plt.ylabel("Actual Label",     fontsize=11)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print("✓ Confusion matrix saved as confusion_matrix.png")

In [ ]:
from IPython.display import display

print("=" * 70)
print("   FIGURE 3: Classification Results")
print("=" * 70)

# Show 15 sample results
results = df[["subreddit","title","sentiment",
              "sentiment_score","topic",
              "engagement_level"]].head(15)

pd.set_option("display.max_colwidth", 40)
display(results)

# Summary stats
print("\n── Sentiment Distribution ──────────────────")
print(df["sentiment"].value_counts().to_string())

print("\n── Topic Distribution ──────────────────────")
print(df["topic"].value_counts().to_string())

print("\n── Engagement Distribution ─────────────────")
print(df["engagement_level"].value_counts().to_string())

print(f"\n── Accuracy Score : {acc:.2%} ──────────────────")

In [ ]:
# Save final CSV
df.to_csv("classified_reddit_data.csv", index=False, encoding="utf-8-sig")

# Download both files
from google.colab import files
files.download("classified_reddit_data.csv")
files.download("confusion_matrix.png")

print(f"✓ classified_reddit_data.csv saved")
print(f"   Rows    : {df.shape[0]}")
print(f"   Columns : {df.shape[1]}")
print(f"\n   New columns added:")
print(f"   → sentiment")
print(f"   → sentiment_score")
print(f"   → topic")
print(f"   → engagement_level")
print(f"   → ground_truth")
print(f"\n✓ confusion_matrix.png saved")